In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import numpy as np

In [2]:
# Check data hadoop
PATH_TO_CSV = "../../hdfs/data-input/drug200.csv"
pass_linux = 'echo 123 | sudo -S'

In [3]:
# # Update data in HDFS

# !docker exec namenode hdfs dfs -rm -r -f /user/data

# !docker exec namenode hdfs dfs -mkdir -p /user/data

# !docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.csv

# !docker exec namenode hdfs dfs -put /tmp/drug200.csv /user/data/

# # Check result
# !docker exec namenode hdfs dfs -ls -R /user/data/

In [4]:
check_data = !{pass_linux} docker exec namenode hdfs dfs -ls -R /user/data/
if not check_data:
    !{pass_linux} docker exec namenode hdfs dfs -mkdir -p /user/data
    !{pass_linux} docker cp "{PATH_TO_CSV}" namenode:/tmp/drug200.csv
    !{pass_linux} docker exec namenode hdfs dfs -put /tmp/drug200.csv /user/data/
    !{pass_linux} docker exec namenode hdfs dfs -ls -R /user/data/
else:
    print("Dữ liệu đã tồn tại trên HDFS.")

Dữ liệu đã tồn tại trên HDFS.


In [ ]:
memory_use = "20g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 15:51:17 WARN Utils: Your hostname, lenovo-Legion-5-15IAH7H, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/04/24 15:51:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 15:51:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# Read from HDFS
ip_namenode = '172.20.0.2'
spark.sparkContext.setCheckpointDir(f"hdfs://{ip_namenode}:9000/user/checkpoints")
hdfs_path = f"hdfs://{ip_namenode}:9000/user/data/drug200.csv"
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)
# Check load data
df = df.repartition(500) 
df.show(5)
df.printSchema()

+---+---+------+-----------+-------+-----+
|Age|Sex|    BP|Cholesterol|Na_to_K| Drug|
+---+---+------+-----------+-------+-----+
| 21|  M|   LOW|     NORMAL|  17.71|DrugY|
| 30|  F|NORMAL|       HIGH|  7.007|DrugX|
| 53|  F|NORMAL|     NORMAL| 10.056|DrugX|
| 72|  M|  HIGH|     NORMAL| 33.098|DrugY|
| 29|  M|   LOW|       HIGH| 29.945|DrugY|
+---+---+------+-----------+-------+-----+
only showing top 5 rows
root
 |-- Age: integer (nullable = true)
 |-- Sex: string (nullable = true)
 |-- BP: string (nullable = true)
 |-- Cholesterol: string (nullable = true)
 |-- Na_to_K: double (nullable = true)
 |-- Drug: string (nullable = true)



In [7]:
# Calculate Hardware
def get_system_usage():
    cpu_percent = psutil.cpu_percent(interval=None)
    ram_usage = psutil.virtual_memory().percent
    return cpu_percent, ram_usage

start_time = time.time()
cpu_start, ram_start = get_system_usage()

total_rows = df.count()

end_time = time.time()
cpu_end, ram_end = get_system_usage()

In [8]:
# Size data
total_cols = len(df.columns)

stats = df.select(
    F.mean("Age").alias("Average_Age"),
    F.percentile_approx("Na_to_K", 0.5).alias("Median_Na_to_K")
).collect()[0]

In [9]:
# Result calculate data
print(f"Kích thước: {total_rows} hàng x {total_cols} cột")
print(f"Trung bình Age: {stats['Average_Age']:.2f}")
print(f"Trung vị chỉ số (Na_to_K): {stats['Median_Na_to_K']}")

# Performance CPU RAM
print(f"Thời gian truy xuất (Latency): {end_time - start_time:.2f} giây")
print(f"Mức độ CPU sử dụng: {cpu_end}%")
print(f"Mức độ RAM sử dụng: {ram_end}%")

# Number of partitions
num_partitions = df.rdd.getNumPartitions()
print(f"Số lượng partitions của DataFrame: {num_partitions}")

# Distribution
partition_counts = df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count()

partition_counts.show()

Kích thước: 50000000 hàng x 6 cột
Trung bình Age: 44.50
Trung vị chỉ số (Na_to_K): 20.501
Thời gian truy xuất (Latency): 4.66 giây
Mức độ CPU sử dụng: 91.0%
Mức độ RAM sử dụng: 40.1%


Số lượng partitions của DataFrame: 500


+------------+-----+
|partition_id|count|
+------------+-----+
|           0|99995|
|           1|99994|
|           2|99995|
|           3|99994|
|           4|99995|
|           5|99994|
|           6|99994|
|           7|99993|
|           8|99993|
|           9|99993|
|          10|99993|
|          11|99993|
|          12|99992|
|          13|99993|
|          14|99995|
|          15|99995|
|          16|99995|
|          17|99995|
|          18|99995|
|          19|99996|
+------------+-----+
only showing top 20 rows


In [10]:
# Data preprocessing
# Feature Engineering
# Indexing Label
labelIndexer = StringIndexer(inputCol="Drug", outputCol="indexedLabel", handleInvalid="skip")

# Indexing to classification fields
sexIndexer = StringIndexer(inputCol="Sex", outputCol="Sex_Index")
bpIndexer = StringIndexer(inputCol="BP", outputCol="BP_Index")
cholIndexer = StringIndexer(inputCol="Cholesterol", outputCol="Cholesterol_Index")

# gom các đặc trưng vào một vector
featureCols = ["Age", "Sex_Index", "BP_Index", "Cholesterol_Index", "Na_to_K"]
assembler = VectorAssembler(inputCols=featureCols, outputCol="features")

rf = RandomForestClassifier(
    labelCol="indexedLabel", 
    featuresCol="features", 
    numTrees=80,           
    maxDepth=12,           
    maxBins=64,          
    subsamplingRate=0.7,  
    seed=42  
)

# Pipeline
pipeline = Pipeline(stages=[labelIndexer, sexIndexer, bpIndexer, cholIndexer, assembler, rf])

# Split Train/Test
(trainingData, testData) = df.randomSplit([0.8, 0.2], seed=42)

df.unpersist()

# Training
model = pipeline.fit(trainingData)

# Evaluate
predictions = model.transform(testData)

# Calculate Confusion Matrix by RDD API
predictionAndLabels = predictions.select("prediction", "indexedLabel") \
                                 .rdd.map(lambda row: (float(row.prediction), float(row.indexedLabel)))

evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction")

# Accuracy
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})

# Weighted F1-Score (Trong Spark f1 chính là weighted F1)
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

# Weighted Precision
weighted_precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})

# Weighted Recall
weighted_recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {weighted_precision:.4f}")
print(f"Recall: {weighted_recall:.4f}")

metrics = MulticlassMetrics(predictionAndLabels)
print("Confusion Matrix:")
cm= metrics.confusionMatrix().toArray()
print(cm)

26/04/24 15:58:31 WARN MemoryStore: Not enough space to cache rdd_139_406 in memory! (computed 30.7 MiB so far)
26/04/24 15:58:31 WARN BlockManager: Persisting block rdd_139_406 to disk instead.
26/04/24 15:58:32 WARN MemoryStore: Not enough space to cache rdd_139_408 in memory! (computed 8.6 MiB so far)
26/04/24 15:58:32 WARN BlockManager: Persisting block rdd_139_408 to disk instead.
26/04/24 15:58:32 WARN MemoryStore: Not enough space to cache rdd_139_410 in memory! (computed 3.8 MiB so far)
26/04/24 15:58:32 WARN BlockManager: Persisting block rdd_139_410 to disk instead.
26/04/24 15:58:32 WARN MemoryStore: Not enough space to cache rdd_139_413 in memory! (computed 3.8 MiB so far)
26/04/24 15:58:32 WARN BlockManager: Persisting block rdd_139_413 to disk instead.
26/04/24 15:58:33 WARN MemoryStore: Not enough space to cache rdd_139_414 in memory! (computed 19.5 MiB so far)
26/04/24 15:58:33 WARN BlockManager: Persisting block rdd_139_414 to disk instead.
26/04/24 15:58:33 WARN Memor

Accuracy: 0.9993
F1-Score: 0.9993
Precision: 0.9993
Recall: 0.9993


Confusion Matrix:


[[6.888068e+06 2.292000e+03 2.367000e+03 1.383000e+03 9.520000e+02]
 [0.000000e+00 1.033690e+06 0.000000e+00 0.000000e+00 0.000000e+00]
 [0.000000e+00 0.000000e+00 1.033650e+06 0.000000e+00 0.000000e+00]
 [0.000000e+00 0.000000e+00 0.000000e+00 6.220800e+05 0.000000e+00]
 [0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 4.151020e+05]]


In [11]:
# Save model
model.write().overwrite().save("model_random_forest_spark")

In [12]:
import gc

In [13]:
# Free RAM
predictions.unpersist()
testData.unpersist()
trainingData.unpersist()
del predictions
del metrics
gc.collect()
spark.catalog.clearCache()

In [ ]:
# from IPython.display import display, HTML
# display(HTML("<script>Jupyter.notebook.kernel.restart()</script>"))

: 

In [ ]:
# Stop
spark.stop()
# !{pass_linux} shutdown now
import os
os._exit(0)